# Week 08 — Modelling and Independent Evaluation

**Spatial Data Science with Python · MSc GIS / AGMS 1333**  
**Official Table 4 topic:** Model Selection and Evaluation · Holdout testing, MAE, RMSE, bias, R²

**Learning outcome.** Evaluate the frozen model on an eastern test strip.

**What you will implement:** Train/test partition, error metrics, residual map.  
**Mathematical model:** RMSE = sqrt(Σ(ŷ−y)²/n); bias = mean(ŷ−y).

> **Data provenance:** The Urban Teaching Grid is a **synthetic** 12 × 12 grid at 250-m cell spacing. Its temperatures and facilities are simulated; do not present its numerical outputs as field measurements. All distance calculations in the labs use the projected coordinates EPSG:32647 where relevant.

**How to run.** In Colab, upload this notebook from *File → Upload notebook*. Run each cell top-to-bottom. The setup cell below will ask for `DATA_ONLY.zip` only if the data folder is absent. In JupyterLab, extract the complete ZIP and open this notebook in `01_Core_Labs/`.


## 1. Prepare the environment and locate the data

The same setup works in JupyterLab and Google Colab. It selects an existing local data directory, or prompts for the packaged dataset in Colab. `assert` fails early if the archive is incomplete.

In [ ]:
from pathlib import Path
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Local Jupyter: data/ or ../data. Colab: upload DATA_ONLY.zip once per runtime.
candidates = [Path('data'), Path('../data'), Path('/content/data')]
DATA = next((p for p in candidates if (p / 'cells.csv').is_file()), None)
if DATA is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Cannot find data/. Extract the course ZIP and keep the data folder with the notebooks.') from exc
    print('Select DATA_ONLY.zip from the downloaded course package:')
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.endswith('.zip')), None)
    if zip_name is None:
        raise FileNotFoundError('Please upload DATA_ONLY.zip and run this cell again.')
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
        archive.extractall(Path.cwd())
    DATA = Path('data')
assert (DATA / 'metadata.json').is_file(), 'Data package incomplete.'
print('Dataset folder:', DATA.resolve())
print('Synthetic teaching data: not field observations.')


## 2. Key concepts and expected data

The analysis follows **input → cleaning → computation → validation → interpretation**. Always inspect observation keys, coordinate units, missing values, and the intended study period before fitting a model.

**Mathematical foundation:** RMSE = sqrt(Σ(ŷ−y)²/n); bias = mean(ŷ−y)

**Method family:** Train/test partition, error metrics, residual map.

## 3. Main computation

Run this cell after the setup. The block calculates the principal statistics or fits the weekly model. The original course code is retained so results remain aligned with the slide examples.

In [ ]:
from sklearn.model_selection import GroupKFold,cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
cells=pd.read_csv(DATA/'cells.csv');obs=pd.read_csv(DATA/'observations_clean.csv');june=cells.merge(obs[obs.date=='2025-06-01'],on='cell_id')
FEATURES=['canopy_frac','impervious_frac','elevation_m'];dev=june[june.col<9];east=june[june.col>=9]
cv=GroupKFold(n_splits=4)
candidates={'Mean':DummyRegressor(),**{f'Ridge α={alpha}':Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler()),('reg',Ridge(alpha=alpha))]) for alpha in [.1,1,10]}}
score={k:-cross_val_score(v,dev[FEATURES],dev.lst_c,groups=dev.block_id,cv=cv,scoring='neg_mean_absolute_error').mean() for k,v in candidates.items()}
name=min(score,key=score.get);model=candidates[name].fit(dev[FEATURES],dev.lst_c)
pred=model.predict(east[FEATURES]);err=pred-east.lst_c.to_numpy()
print('Selected on development:',name)
print('East MAE',mean_absolute_error(east.lst_c,pred),'RMSE',np.sqrt(mean_squared_error(east.lst_c,pred)),'Bias',err.mean(),'R²',r2_score(east.lst_c,pred))

### Reading the main code

**Freeze and refit.** Use the model-selection procedure on development geography only; after selecting, refit on all development data.

- `GroupKFold` — Keeps the same spatial group within one validation fold.
- `Pipeline` — Combines transformations and estimator, fitting preprocessing only on training rows.
- `SimpleImputer` — Replaces missing predictors with a statistic learned on training data.
- `StandardScaler` — Centers and scales columns using values learned from training data.
- `pd.read_csv` — Reads CSV text into a pandas DataFrame. Inspect head() and column types after loading.
- `cross_val_score` — Computes validation scores under a specified cross-validation scheme.

**Verification questions:** Which variables contain the inputs? What new object does the cell create? What numerical result should be checked before continuing?

## 4. Map, plot, or evaluate the result

The final block turns results into a labelled graphic or output artifact. Keep units in axes/legends, and distinguish model predictions from observations.

In [ ]:
plt.scatter(east.col,east.row,c=err,s=165,marker='s',cmap='coolwarm');plt.colorbar(label='Predicted - reference (°C)')
plt.title('Independent eastern strip');plt.xlabel('Grid column');plt.ylabel('Grid row');plt.show()

### Reading the output code

**Independent test.** East targets are used once for evaluation. MAE, RMSE, bias, R² and training-mean baseline answer different evaluation questions.

- `plt.` — Creates a figure; title, axes, and legend are part of the analytical interpretation.

**Spatial residual map.** Map predicted minus observed values so systematic transfer problems can be seen geographically.

## 5. Critical interpretation

### Interpret your results

Complete the following short report **after** running the notebook:

1. **Research question:** What geographic question does this notebook address?
2. **Observation unit and period:** What does one row/pixel represent? What is its date and CRS?
3. **Method:** Why is this method suitable, and what assumptions does it make?
4. **Results:** Report two numerical outputs or figure patterns with units.
5. **Validity:** State one data limitation and one model/evaluation limitation.
6. **Extension:** Modify one parameter, compare results, and describe what changed.


## 6. Questions and independent practice

1. Map signed errors by grid block and discuss where the selected model underpredicts or overpredicts.
2. What would happen if the spatial unit, study date, or CRS changed?
3. Is this analysis descriptive, inferential, predictive, or a decision-support example? Justify your answer.
4. Which assumption should be tested before use in a real project?

**Next step:** Open the matching advanced notebook in `02_Advanced_Experiments/` for four short method variations.